# Fine-Tune Mistral-7B-v0.3 with LoRA on Docker Q&A

**Task #5** of the weekend plan: the first fine-tuning run (LoRA r=16). We load the **base** Mistral-7B in 4-bit, attach LoRA adapters, train for 3 epochs on `data/train/train_1k.jsonl`, and save the adapter so Sunday's eval can score it.

**Runtime budget**: ~1–2 hours on a free Colab T4.

**Before you run this notebook, make sure:**
1. Runtime → Change runtime type → **T4 GPU** is selected.
2. You've accepted Mistral-7B's license on HuggingFace (it's gated).
3. `HF_TOKEN` is set in Colab Secrets (🔑 left sidebar). `WANDB_API_KEY` is optional.
4. The latest commits are pushed to the GitHub repo cloned below — Colab pulls a fresh copy each run.

## 1. Confirm GPU is available

Should print a T4 with ~15 GB VRAM. If `nvidia-smi` fails or shows no GPU, switch the runtime type and restart.

In [1]:
!nvidia-smi

Mon Jun 22 16:13:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone the project

Replace `<your-username>/LLM_Finetune` with your actual repo. For a private repo, use `https://<token>@github.com/<you>/LLM_Finetune.git` with a fine-grained PAT that has read access.

In [2]:
!git clone https://github.com/Gabriel-Kevorkian/LLM_Finetune.git
%cd LLM_Finetune

Cloning into 'LLM_Finetune'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 97 (delta 37), reused 86 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (97/97), 460.22 KiB | 2.35 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/LLM_Finetune


## 3. Install training dependencies

Compared to the baseline notebook we additionally need:
- **`unsloth`** — fast LoRA training kernels (~2× faster than vanilla on T4).
- **`trl`** — HuggingFace's TRL library, provides `SFTTrainer`.
- **`peft`** — implements LoRA (Unsloth wraps it).
- **`bitsandbytes`** — provides 4-bit quantization + 8-bit AdamW optimizer.
- **`accelerate`** — handles device placement / mixed precision.

**If this cell errors out** with version conflicts, check the current recommended install line at <https://github.com/unslothai/unsloth#installation> — the Unsloth + Colab combo gets updated frequently and pinning can become stale within weeks.

Install takes ~2 minutes.

In [3]:
%%capture
!pip install -q -U unsloth
!pip install -q -U trl peft accelerate bitsandbytes datasets
!pip install -q -r requirements.txt

## 4. Load secrets

We read tokens from the Colab Secrets panel into env vars. `HF_TOKEN` is required to download the gated Mistral weights. `WANDB_API_KEY` is optional — if present, we enable Weights & Biases logging on the training run.

In [4]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"]               = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = userdata.get("HF_TOKEN")

try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    USE_WANDB = True
except Exception:
    USE_WANDB = False

print(f"HF_TOKEN:       {'set' if os.environ.get('HF_TOKEN') else 'MISSING'}")
print(f"WANDB_API_KEY:  {'set' if USE_WANDB else 'not set (W&B disabled)'}")

HF_TOKEN:       set
WANDB_API_KEY:  set


## 5. Run the fine-tune

This will:
1. Download the Unsloth pre-quantized Mistral-7B 4-bit weights (~4 GB, ~1 minute).
2. Attach LoRA adapters (r=16 by default — see `configs/base.yaml`).
3. Format each row of `data/train/train_1k.jsonl` into Mistral's `[INST]...[/INST]` chat format.
4. Train for 3 epochs (~375 optimizer steps at effective batch size 8).
5. Save the LoRA adapter (~80 MB) to `models/adapters/r16/`.
6. Save a training summary to `results/runs/r16/`.

**Loss expectations**: starts around 1.5–2.0, should drop to ~0.5–0.8 by end of training. If it stays >1.5 the whole run, something is off (check the chat-template example printed at the top — the `[INST]...[/INST]` markers should be visible).

In [5]:
wandb_flag = "--wandb" if USE_WANDB else ""
!python scripts/04_train.py --config configs/base.yaml {wandb_flag}

Fine-tune run: r16
  base model:  unsloth/mistral-7b-v0.3-bnb-4bit
  train file:  /content/LLM_Finetune/data/train/train_1k.jsonl
  rank/alpha:  16/16
  lr:          0.0002
  epochs:      3
  batch:       2 × 4 grad-accum = 8 effective
  seq_len:     2048
  wandb:       True
  seed:        42

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading base model: unsloth/mistral-7b-v0.3-bnb-4bit
==((====))== 

## 6. Inspect the saved adapter

After training we have two output locations:
- `models/adapters/r16/` — the LoRA weights + tokenizer config. This is what `scripts/05_eval_finetuned.py` loads on Sunday.
- `results/runs/r16/` — `config.yaml` (frozen snapshot of hyperparameters) + `train_summary.json`.

The adapter folder should be ~80 MB total — a tiny fraction of the 14 GB base model.

In [6]:
!ls -lh models/adapters/r16/
print()
!cat results/runs/r16/train_summary.json

ls: cannot access 'models/adapters/r16/': No such file or directory

cat: results/runs/r16/train_summary.json: No such file or directory


In [9]:
!pwd
!ls -la models/adapters/r16/

/content/LLM_Finetune
ls: cannot access 'models/adapters/r16/': No such file or directory


## 7. Smoke test — generate one answer from the fine-tuned model

Quick visual check that the adapter actually changed the model's behaviour. We pick an eval question and generate an answer with the freshly-trained model.

**This is NOT the formal eval** (that's Sunday's job, scripts/05). It's a sanity check that:
1. The adapter loaded correctly,
2. The model produces Docker-flavoured answers,
3. The output looks better than the baseline rambles we saw earlier.

In [7]:
import sys
sys.path.insert(0, ".")

from unsloth import FastLanguageModel
from src.data.format_prompts import format_chat, ensure_chat_template

# Load base + the trained adapter in one shot. Unsloth detects the adapter
# config in the folder and merges it onto the base weights automatically.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "models/adapters/r16",
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)
ensure_chat_template(tokenizer)
# Switch into Unsloth's optimized inference mode (disables dropout, etc.)
FastLanguageModel.for_inference(model)

question = "What is the difference between COPY and ADD in a Dockerfile?"
prompt   = format_chat(question, tokenizer)
inputs   = tokenizer(prompt, return_tensors="pt").to("cuda")

out_ids  = model.generate(**inputs, max_new_tokens=256, do_sample=False)
response = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Q:", question)
print("A:", response)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


RuntimeError: Unsloth: No config file found - are you sure the `model_name` is correct?
If you're using a model on your local device, confirm if the folder location exists.
If you're using a HuggingFace online model, check if it exists.

## 8. Persist the adapter so Sunday's eval can find it

`/content/` on Colab is **wiped when the runtime disconnects**. If you plan to come back tomorrow for Sunday's eval, you need to save the adapter somewhere persistent.

**Option A — Save to Google Drive (recommended).** Run the next cell. Sunday's notebook re-mounts the same Drive folder.

**Option B — Download a zip locally** (use if you don't want to mount Drive). Drag the resulting zip out of the file browser, then re-upload it tomorrow before Sunday's eval.

In [ ]:
# Option A: copy adapter + results to Google Drive
from google.colab import drive
drive.mount("/content/drive")

import shutil, pathlib
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/LLM_Finetune")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Mirror models/adapters/r16/ and results/runs/r16/ into Drive
shutil.copytree("models/adapters/r16", DRIVE_ROOT / "models/adapters/r16", dirs_exist_ok=True)
shutil.copytree("results/runs/r16",    DRIVE_ROOT / "results/runs/r16",    dirs_exist_ok=True)
print(f"Adapter + run dir saved to {DRIVE_ROOT}")

In [ ]:
# Option B (alternative): zip + download to local machine
# Uncomment to use.
#
# import shutil
# shutil.make_archive("r16_adapter", "zip", "models/adapters/r16")
# from google.colab import files
# files.download("r16_adapter.zip")

## 9. Commit the small artifacts back to git

The adapter itself is gitignored (`models/` excluded by `.gitignore`). But the **frozen config + training summary** in `results/runs/r16/` are tiny and reproducible — those go to git so the README's results table has an audit trail.

**You will git-push from your local machine after this notebook.** That is, copy `results/runs/r16/config.yaml` and `results/runs/r16/train_summary.json` to your local repo (via Drive or download), then `git add` + `git commit` + `git push` locally. We deliberately avoid pushing from Colab to keep the project's git history clean (no `colab` author commits).